#### Import packages

In [1]:
import altair as alt
import pandas as pd

Import `publication-grids.csv` maintained by [Jane Pong](github.com/officeofjane)

Starting point [How to create a US Grid Cartogram using Altair](https://medium.com/@parham.motameni/how-to-create-a-us-grid-cartogram-using-altair-1800e8513639) posted by Parham Motameni.

In [2]:
url = 'https://gist.githubusercontent.com/officeofjane/2c3ed88c4be050d92765de912d71b7c4/raw/8abc5e19a5fac2ec63e81891d9bfdebe2d55f92c/publication-grids.csv'
df = pd.read_csv(url)

I like the grid for NPR - it includes DC and the placement of each grid seems best to me.

In [3]:
df = df[df['publication'] == 'NPR']

#### Reading in my own data

In [5]:
# csv = pd.read_csv("data/50 states and abbreviations - 20250714.csv")
csv = pd.read_csv("data/50_states_days.csv")

Merge CSV with state grid dataframe on state abbreviations.

In [6]:
df = df.merge(csv, left_on='code', right_on='Abbreviation', how='left')

#### Create `heatmap` object

The following is my first call to `Altair` to create a chart. I will use the `mark_bar()` method to create a bar chart.   
The `encode()` method is used to specify the x and y axes of the chart.   
The `x` axis will be the state, and the `y` axis will be the # of days. The `color` parameter is used to color the bars based on # of days.

Introduce `magma` color scheme

In [7]:
heatmap = alt.Chart(df).mark_rect(stroke=None, cornerRadius=5).encode(
    x=alt.X('col:O', axis=None),
    y=alt.Y('row:O', axis=None),
    color=alt.Color(
        'days:Q', 
        scale=alt.Scale(
            type='log', 
            domain=[0.1, 6000], 
            scheme='magma', 
            reverse=True),
        legend=alt.Legend(
            title='Days (log)',
            gradientLength=200,
            labelFontSize=12,
            titleFontSize=14
        )),
    tooltip=['code', 'days']
).properties(
    width=600,
    height=400,
    title='State-Level Days Heatmap (Log Scale)'
)

In [8]:

# Create a base chart for text layers without color encoding
base_text = alt.Chart(df).encode(
    x=alt.X('col:O', axis=None),
    y=alt.Y('row:O', axis=None)
)

# Create fake text outline using small offsets in 4 directions
offsets = [(-2, 0), (2, 0), (0, -2), (0, 2)]

shadow_layers = [
    base_text.mark_text(
    # heatmap.mark_text(
        align='center',
        baseline='middle',
        fill='black',
        opacity=0.4,
        fontSize=12,
        fontWeight='bold',
        dx=dx,
        dy=dy
    ).encode(
        text='code:O'
    )
    for dx, dy in offsets
]

# Foreground white text layer
text_foreground = base_text.mark_text( #heatmap.mark_text(
    align='center',
    baseline='middle',
    # color='white',
    fill='white',
    fontSize=12,
    # fontWeight='bold'
).encode(
    text='code:O'
)

Light gray background must be introduced later because I cannot combine Objects with a 'config' attribute.

In [9]:
# Combine all layers
chart = alt.layer(
    heatmap,
    *shadow_layers,
    text_foreground
).properties(
    width=600,
    height=400,
    title="Where I've spent my days"
).configure_view(
    stroke=None,
    fill='#d9d9d9'  # Set the background color to light gray
).resolve_scale(
    color='independent'
)

chart.display()


alt.LayerChart(...)